In [3]:
import json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from apple_transform import NUMERIC_FEATURES, transformed_name

# URL untuk REST API TensorFlow Serving
SERVER_URL = 'http://localhost:8501/v1/models/apple_quality_model:predict'

# Fungsi untuk melakukan preprocessing data
def preprocess_data(data):
    # Normalisasi data menggunakan mean dan std yang sama dengan yang digunakan saat training
    # Catatan: Dalam produksi sebenarnya, nilai ini harus diambil dari transform_fn
    
    # Contoh data yang sudah dinormalisasi (simulasi)
    processed_data = {}
    
    # Proses fitur numerik
    for feature in NUMERIC_FEATURES:
        processed_data[transformed_name(feature)] = [float(data[feature])]
    
    # Proses Acidity
    processed_data[transformed_name('Acidity')] = [float(data['Acidity'])]
    
    return processed_data

# Fungsi untuk melakukan prediksi
def predict_quality(data):
    # Preprocess data
    processed_data = preprocess_data(data)
    
    # Format data untuk TensorFlow Serving
    instances = {
        'instances': [processed_data]
    }
    
    # Kirim request ke server
    response = requests.post(SERVER_URL, json=instances)
    
    # Parse response
    result = json.loads(response.text)
    prediction = result['predictions'][0][0]
    
    # Konversi probability ke label
    label = "good" if prediction > 0.5 else "bad"
    probability = prediction if prediction > 0.5 else 1 - prediction
    
    return {
        "label": label,
        "probability": probability,
        "raw_prediction": prediction
    }

# Contoh penggunaan
sample_apple = {
    "A_id": 1,
    "Size": 0.5,
    "Weight": 0.6,
    "Sweetness": 0.7, 
    "Crunchiness": 0.8,
    "Juiciness": 0.9,
    "Ripeness": 0.7,
    "Acidity": "0.5"
}

# Lakukan prediksi
try:
    result = predict_quality(sample_apple)
    print(f"Prediction: {result['label']}")
    print(f"Probability: {result['probability']:.4f}")
    print(f"Raw prediction value: {result['raw_prediction']:.4f}")
except Exception as e:
    print(f"Error during prediction: {e}")
    print("Note: Make sure TensorFlow Serving is running with the correct model.")
    print("Run the following command to start TensorFlow Serving:")
    print("docker build -t apple-quality-serving . && docker run -p 8501:8501 -p 8500:8500 apple-quality-serving")

# Visualisasi hasil prediksi
def visualize_prediction(result):
    labels = ['Bad', 'Good']
    probabilities = [1 - result['raw_prediction'], result['raw_prediction']]
    
    plt.figure(figsize=(8, 6))
    plt.bar(labels, probabilities, color=['#FF9999', '#99FF99'])
    plt.ylim(0, 1)
    plt.title('Apple Quality Prediction')
    plt.ylabel('Probability')
    
    # Add text labels
    for i, prob in enumerate(probabilities):
        plt.text(i, prob + 0.05, f'{prob:.2f}', ha='center')
    
    plt.show()

# Tampilkan visualisasi jika prediksi berhasil
try:
    visualize_prediction(result)
except Exception as e:
    print(f"Could not visualize: {e}")


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject